# 25 · Kiểm cờ xoay EXIF trong OCR cache ViTextVQA (W14, bước 1 — luật R-EXIF)

**Vấn đề (W13, bước 8):** ở `1849.jpg` và `14277.jpg` (4000×3000), cache OCR chứa chữ không liên quan (`WHIRS`, `Transly`, …), còn OCR trên ảnh đã xoay / thu nhỏ đọc đúng biển hiệu. Nếu lỗi này có hệ thống thì OCR mà B3 / B3-full / SH dùng lúc train **và** lúc test bị hỏng ở một phần dữ liệu.

**Cơ chế nghi ngờ:** trong `src/data/ocr.py`, Paddle **dò hộp chữ** từ đường dẫn file (đọc bằng OpenCV, OpenCV **tự xoay** theo cờ EXIF), còn VietOCR **cắt hộp** từ ảnh PIL (PIL **không xoay**). Ảnh có cờ xoay → hộp và ảnh lệch nhau → cắt nhầm vùng → chữ rác.

**Notebook này làm (không đổi cache, không đổi code OCR, không đổi số đã báo cáo):**
1. Minh hoạ cơ chế trên một ảnh giả (mục 2).
2. **Bước A** — quét cờ EXIF của mọi ảnh ViTextVQA + ViVQA, đếm câu hỏi bị ảnh hưởng theo split (mục 4, CPU).
3. **Bước B** — OCR lại mẫu cố định (≤ 300 ảnh có cờ xoay + 200 ảnh đối chứng, seed 42, luôn có 2 ảnh nghi vấn) theo 3 kiểu `asis` / `fixed` / `small`, so với cache (mục 6).
4. **Bước C** — áp luật L1–L3 đã ghi trước ở `docs/W14/README.md` (mục 7).

**Runtime:** **T4 GPU** (Runtime → Change runtime type → T4 GPU). Không nạp Qwen. Thời gian dự kiến 45–70 phút (giải nén ảnh ~10 phút, cài Paddle ~5 phút, OCR lại ~20–40 phút). Cuối notebook **tự trả máy**.

**Anh chỉ cần:** Run all → đợi → gửi Claude **output mục 8** (một khối chữ). Nếu một cell dừng với dấu ❌ thì gửi output cell đó.

## 1. Thiết lập môi trường

Gắn Google Drive, lấy code mới nhất từ GitHub, in loại máy.

*Kỳ vọng: commit mới nhất có `src/data/exif_audit.py`, `Runtime: Tesla T4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, subprocess
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('src/data/exif_audit.py'), '❌ Code cũ -> git pull chưa lấy được commit W14 bước 1'
assert 'R-EXIF' in open('docs/W14/README.md', encoding='utf-8').read(), '❌ Luật R-EXIF chưa được commit -> DỪNG'

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
OUT = f'{DRIVE_ROOT}/experiments/W14_exif_audit_vitextvqa'
os.makedirs(OUT, exist_ok=True)
CACHE = f'{DRIVE_ROOT}/data/vitextvqa_official/ocr/paddledet_vietocr.json'
CACHE_SHA_W10 = 'e7ce4f76d98c1eb5dbbd7c07bb6ed62cfb3e4043b955143a2b5a6cc182560c1b'   # biên bản W10 (15.352 ảnh)

GPU = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip() or 'CPU'
print('Runtime:', GPU, '(T4 là đủ; CPU vẫn chạy được nhưng mục 6 chậm hơn khoảng 3 lần)')
print('✅ Thiết lập xong')

## 2. Minh hoạ cơ chế (ảnh giả, không cần dữ liệu)

Tạo một ảnh 400×300 có cờ EXIF "xoay 90°" rồi đọc bằng hai thư viện.

*Kỳ vọng: `OpenCV đọc ra (300, 400)` và `PIL đọc ra (400, 300)` → hai thư viện **không nhìn cùng một ảnh**. Nếu Paddle dò chữ trên ảnh OpenCV còn VietOCR cắt trên ảnh PIL thì hộp chữ rơi sai chỗ.*

In [ ]:
import cv2
from PIL import Image
im = Image.new('RGB', (400, 300), 'white'); ex = Image.Exif(); ex[274] = 6
im.save('/content/_exif_demo.jpg', exif=ex)
h, w = cv2.imread('/content/_exif_demo.jpg').shape[:2]
print('OpenCV', cv2.__version__, 'đọc ra', (w, h))
print('PIL đọc ra', Image.open('/content/_exif_demo.jpg').size)

## 3. Kiểm thử offline + đưa ảnh về máy Colab

- Selftest của `exif_audit` (không cần GPU, không cần dữ liệu).
- Giải nén ảnh ViTextVQA (zip trên Drive) và chép ảnh ViVQA về `/content`, giống các notebook trước. Chỉ làm một lần mỗi phiên.

*Kỳ vọng: `exif_audit SELFTEST OK`, `ocr SELFTEST OK`, hai dòng thư mục ảnh. Lần đầu mất khoảng 10 phút.*

In [ ]:
!python -m src.data.exif_audit --selftest
!python -m src.data.ocr --selftest 2>&1 | tail -1
from src.eval.run_baseline import prepare_images
IMG_TV = prepare_images('vitextvqa_official', DRIVE_ROOT)
IMG_VV = prepare_images('vivqa', DRIVE_ROOT)
print('ảnh ViTextVQA:', IMG_TV, '| ảnh ViVQA:', IMG_VV)

## 4. Bước A — quét cờ EXIF (chỉ đọc header ảnh, CPU)

Với mỗi split: số câu, số ảnh, số ảnh có cờ xoay (Orientation ≠ 1) và **số câu hỏi** dùng các ảnh đó. Sau đó xem riêng 2 ảnh nghi vấn: kích thước theo PIL và theo OpenCV.

*Kỳ vọng: bảng 5 dòng (ViTextVQA train / dev / test, ViVQA train / test). Cột `% câu EXIF` của ViTextVQA test và train là đầu vào của luật L2 / L3. Dòng 2 ảnh nghi vấn: nếu `o=6` (hoặc 5/7/8) và OpenCV đọc ra kích thước đảo so với PIL → cơ chế rất có thể đúng với 2 ảnh này.*

In [ ]:
!python -m src.data.exif_audit --scan --drive_root {DRIVE_ROOT} --vitextvqa_images {IMG_TV} --vivqa_images {IMG_VV} --out {OUT} > {OUT}/scan_log.txt
assert os.path.exists(f'{OUT}/exif_scan.json'), '❌ Bước A lỗi (xem lỗi phía trên) -> gửi Claude output cell này'
SCAN = json.load(open(f'{OUT}/exif_scan.json', encoding='utf-8'))
print(f"{'tập':28s} {'câu':>7s} {'ảnh':>7s} {'ảnh EXIF':>9s} {'câu EXIF':>9s} {'% câu EXIF':>10s}  hướng")
for ds, d in SCAN['summary'].items():
    for sp, s in d.items():
        if sp == 'all_images':
            continue
        print(f"{ds + ' / ' + sp:28s} {s['n_questions']:7d} {s['n_images']:7d} {s['n_images_exif']:9d} "
              f"{s['n_questions_exif']:9d} {s['pct_questions_exif']:10.3f}  {s['exif_by_orientation']}")
for m, v in SCAN['images']['_must_include'].items():
    if not v:
        print(f'{m}: không có trong dữ liệu'); continue
    h, w = cv2.imread(f'{IMG_TV}/{m}').shape[:2]
    print(f"{m}: o={v['o']} | PIL {v['w']}x{v['h']} | OpenCV {w}x{h}")

## 5. Cài PaddleOCR 3.7 + VietOCR (cùng phiên bản với cache) — **torch phải còn nguyên**

Giống hệt notebook 19 (đã chạy ổn ở W11): cài `paddleocr==3.7.0`, thử `paddlepaddle-gpu==3.3.1`, khôi phục NCCL của torch, kiểm Paddle GPU ở tiến trình riêng; không được thì dùng Paddle CPU. Sau đó cài VietOCR `--no-deps`.

*Kỳ vọng: `torch OK trước khi cài`, `paddleocr 3.7.0 | paddle 3.3.1`, `DEVICE = gpu`, `VietOCR import OK`. Mất 3–6 phút.*

In [ ]:
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]

def nvidia_pkgs():
    out = subprocess.run('pip list --format=json', shell=True, capture_output=True, text=True).stdout
    return {p['name'].lower(): p['version'] for p in json.loads(out) if p['name'].lower().startswith('nvidia-')}

def torch_ok():
    return sh('python -c "import torch; assert torch.cuda.is_available(); torch.zeros(1).cuda()"')[0] == 0

def paddle_gpu_ok():
    return sh('python -c "import paddle; assert paddle.device.is_compiled_with_cuda(); paddle.utils.run_check()"')[0] == 0

def restore(pkgs):                       # cài lại ĐÚNG phiên bản cũ, không kéo thêm gì
    if pkgs:
        code_, log = sh('pip install -q --force-reinstall --no-deps ' + ' '.join(f'{n}=={v}' for n, v in pkgs.items()))
        assert code_ == 0, log

HAS_GPU = GPU != 'CPU'
if HAS_GPU:
    assert torch_ok(), '❌ torch đã hỏng từ trước -> Runtime -> Disconnect and delete runtime, rồi chạy lại từ mục 1'
    print('torch OK trước khi cài')
code_, log = sh('pip install -q paddleocr==3.7.0')
assert code_ == 0, log
BEFORE = nvidia_pkgs()
code_gpu = 1
if HAS_GPU:
    for attempt in range(1, 4):
        code_gpu, log_gpu = sh('pip install -q --timeout 180 --retries 5 paddlepaddle-gpu==3.3.1 '
                               '-i https://www.paddlepaddle.org.cn/packages/stable/cu126/')   # = phiên bản cache ViTextVQA
        if code_gpu == 0:
            break
        print(f'⚠ Lần {attempt}/3: cài Paddle GPU lỗi -> thử lại. Đuôi log:', log_gpu[-200:].replace(chr(10), ' '))
    NEW = sorted(set(nvidia_pkgs()) - set(BEFORE))
    restore({n: v for n, v in BEFORE.items() if 'nccl' in n})
    if not torch_ok():
        restore(BEFORE)
if HAS_GPU and code_gpu == 0 and torch_ok() and paddle_gpu_ok():
    DEVICE = 'gpu'
else:
    if HAS_GPU:
        print('⚠ Paddle GPU không dùng được -> gỡ, khôi phục thư viện torch, cài Paddle CPU')
        sh('pip uninstall -y -q paddlepaddle-gpu ' + ' '.join(NEW))
        restore(BEFORE)
    code_, log = sh('pip install -q paddlepaddle==3.3.1')
    assert code_ == 0, log
    DEVICE = 'cpu'
print(sh('python -c "import paddleocr, paddle; print(\'paddleocr\', paddleocr.__version__, \'| paddle\', paddle.__version__)"')[1])
code_, log = sh('pip install -q --no-deps vietocr==0.3.13 && pip install -q gdown einops')
assert code_ == 0, log
print(sh('python -c "from vietocr.tool.predictor import Predictor; print(\'VietOCR import OK\')"')[1])
print('DEVICE =', DEVICE)

## 6. Bước B — OCR lại mẫu cố định, so với cache

**Kiểm trước khi chạy** (sai → dừng):
- cache trên Drive trùng sha256 biên bản W10 (cache chưa bị đổi từ lúc train B3-full);
- engine OCR bây giờ trùng engine đã dựng cache ở các mục quyết định kết quả (`paddleocr`, `det_model`, `rec_model`, `vietocr_config`, `lang`).

**Mỗi ảnh trong mẫu được OCR 3 kiểu:**

| Kiểu | Ảnh đưa vào OCR | Cho biết điều gì |
|---|---|---|
| `asis` | đúng file gốc, như lúc dựng cache | tái hiện cách cũ; với ảnh đối chứng = độ lệch tự nhiên giữa hai lần chạy |
| `fixed` | ảnh đã xoay theo EXIF, lưu PNG không còn cờ | OCR "đúng" — so với cache để biết cache có hỏng không (luật L1) |
| `small` | như `fixed` rồi thu cạnh dài về 2.048 px (giống giao diện W13) | chỉ mô tả — dùng cho bước 3 (sửa đường ảnh của prototype) |

*Kỳ vọng: `cache sha256 trùng biên bản W10 ✅`, `engine trùng cache ✅`, rồi `[R-EXIF] mẫu: … ảnh EXIF≠1 + … ảnh đối chứng`, tiến độ mỗi 25 ảnh, khoảng 20–40 phút trên T4.*

In [ ]:
from src.data.exif_audit import sha256_file
sha = sha256_file(CACHE)
assert sha == CACHE_SHA_W10, f'❌ OCR cache sha256 {sha[:12]}… khác biên bản W10 -> DỪNG, báo Claude'
print('cache sha256 trùng biên bản W10 ✅', sha[:12])

REF = json.load(open('experiments/W10_ocr_cache_fulltrain/ocr_cache_manifest.json', encoding='utf-8'))['engine_info_w10']
rc, out = sh(f'cd /content/ViVQA-VLM && python -c "import json; from src.data.ocr import get_ocr_engine; '
             f'print(json.dumps(get_ocr_engine(\'paddledet_vietocr\', device=\'{DEVICE}\').info))" 2>/dev/null | tail -1')
NOW = json.loads(out.strip().splitlines()[-1])
MUST = ('paddleocr', 'det_model', 'rec_model', 'vietocr_config', 'lang')
bad = {k: (REF.get(k), NOW.get(k)) for k in MUST if REF.get(k) != NOW.get(k)}
assert not bad, f'❌ Engine OCR khác engine đã dựng cache ở {bad} -> DỪNG, báo Claude'
print('engine trùng cache ✅', NOW['det_model'], '+', NOW['rec_model'], '| device', NOW['device'])

!python -u -m src.data.exif_audit --reocr --drive_root {DRIVE_ROOT} --scan_json {OUT}/exif_scan.json --vitextvqa_images {IMG_TV} --cache {CACHE} --expect_cache_sha {CACHE_SHA_W10} --device {DEVICE} --out {OUT} 2>&1 | grep --line-buffered -v "ppocr\|Creating model\|Model files\|UserWarning"
assert os.path.exists(f'{OUT}/exif_reocr.json'), '❌ Bước B chưa ghi được exif_reocr.json -> gửi Claude output cell này'

## 7. Bước C — phán quyết theo luật R-EXIF (ghi trước ở `docs/W14/README.md`)

| Luật | Điều kiện | Hệ quả |
|---|---|---|
| **L1** cơ chế | ≥ 80% ảnh có cờ xoay lệch cache (độ giống < 0,5 với kiểu `fixed`) **và** < 5% ảnh đối chứng lệch | XÁC NHẬN / KHÔNG XÁC NHẬN |
| **L2** test | (chỉ khi L1 xác nhận) > 1% câu full test ViTextVQA có ảnh cờ xoay | sửa `ocr.py`, OCR lại ảnh lỗi, suy luận lại B3-full, báo cả hai số; ≤ 1% → giữ số, ghi giới hạn |
| **L3** train | (chỉ khi L1 xác nhận) > 5% câu train có ảnh cờ xoay | train lại B3-full với cache đã sửa |

*Kỳ vọng: khối JSON `verdict` với `L1`, `L2`, `L3`, rồi bảng "đáp án nằm trong OCR" (cache so với OCR lại) và 2 ảnh nghi vấn.*

In [ ]:
!python -m src.data.exif_audit --analyze --scan_json {OUT}/exif_scan.json --reocr_json {OUT}/exif_reocr.json --out {OUT} > {OUT}/analyze_log.txt
assert os.path.exists(f'{OUT}/exif_audit_summary.json'), '❌ Bước C lỗi -> gửi Claude output cell này'
S = json.load(open(f'{OUT}/exif_audit_summary.json', encoding='utf-8'))
V = S['verdict']
for k in ('L1', 'L2', 'L3'):
    print(f'{k}: {V[k]}')
for g in ('affected', 'control'):
    m = V[f'mismatch_{g}_fixed']
    print(f"lệch cache (fixed) {g:8s}: {m['n_mismatch']}/{m['n']} = {m['pct']}% CI95 {m['ci95']}")
print('\nĐáp án gold nằm trong OCR (câu test, chỉ mô tả):')
for g, d in S['answer_in_ocr_test'].items():
    print(f"  {g:8s} {d['n_questions']:4d} câu | " + ' | '.join(
        f"{t} {d[t]['pct_answer_in_ocr']}%" for t in ('cache', 'asis', 'fixed', 'small') if d[t]['n']))
for m, r in S['descriptive']['must_include'].items():
    print(f'\n{m}: {r}')

## 8. ⭐ Khối gửi Claude

Một khối chữ ngắn chứa mọi số Claude cần để kết luận và viết bước 2. **Copy toàn bộ output cell này gửi Claude.**

*Kỳ vọng: khoảng 25–35 dòng, bắt đầu bằng `=== R-EXIF ===`.*

In [ ]:
print('=== R-EXIF ===')
!git log -1 --format='commit %h'
print('runtime', GPU, '| OCR device', DEVICE)
for ds, d in S['scan_summary'].items():
    for sp, s in d.items():
        if sp != 'all_images':
            print(f"{ds}/{sp}: {s['n_questions_exif']}/{s['n_questions']} câu ({s['pct_questions_exif']}%) | "
                  f"{s['n_images_exif']}/{s['n_images']} ảnh | hướng {s['exif_by_orientation']} | ảnh > 2048 px {s['n_images_gt_ui_max_side']}")
print(json.dumps({k: V[k] for k in ('L1', 'L2', 'L3', 'mismatch_affected_fixed', 'mismatch_control_fixed',
                                    'mismatch_affected_asis', 'mismatch_control_asis',
                                    'test_frac_questions_exif', 'train_frac_questions_exif')}, ensure_ascii=False))
print(json.dumps(S['answer_in_ocr_test'], ensure_ascii=False))
print(json.dumps(S['descriptive'], ensure_ascii=False)[:1500])
R = json.load(open(f'{OUT}/exif_reocr.json', encoding='utf-8'))
for r in [x for x in R['records'] if x['group'] == 'affected'][:3]:
    print(f"vd {r['image']} o={r['o']} sim_fixed={r.get('sim_fixed')} | cache: {r['cache_text'][:70]} | fixed: {r.get('text_fixed', '')[:70]}")
secs = [x.get('sec_fixed', 0) for x in R['records']]
print(f"OCR lại {len(R['records'])} ảnh, TB {sum(secs) / max(len(secs), 1):.2f} s/ảnh (kiểu fixed)")
print('file:', OUT)

## 9. Trả máy

Tự ngắt runtime để không tốn thêm compute. Output phía trên vẫn còn trong tab. Kết quả nằm trên Drive ở `experiments/W14_exif_audit_vitextvqa/` (3 file JSON + 2 log); Cursor sẽ chép lên git ở bước sau.

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')